# Batch inference on AMD

Runs `holdpoint/vision/infer.py` with the checkpoints from `train.ipynb` over the held-out test split of the
Özgenel segmentation set (and the example photos). The output is a cache in the format of the recorded backend:
the app replays it and labels every result "computed on <GPU>, <date>, <command>".

In [ ]:
import os, sys, json, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
from holdpoint.vision import common
DEV = common.pick_device("auto")
INFO = common.device_info(DEV)
print(json.dumps(INFO, indent=2))
# amd-smi is the AMD tool; its output goes next to the results as evidence of the hardware
smi = shutil.which("amd-smi")
SMI = subprocess.run([smi, "static", "--asic", "--json"], capture_output=True, text=True).stdout if smi else None
print(SMI[:800] if SMI else "amd-smi not found")
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")

In [ ]:
RUNS = REPO / "runs"
SEG = Path(os.environ.get("HOLDPOINT_SEG_CKPT") or sorted(RUNS.glob("seg-*/model.pt"))[-1])
CLS = Path(os.environ.get("HOLDPOINT_CLS_CKPT") or sorted(RUNS.glob("cls-*/model.pt"))[-1])
print(SEG, common.sha256_file(SEG)[:12]); print(CLS, common.sha256_file(CLS)[:12])

In [ ]:
from holdpoint.vision import infer
test_cache = REPO / "results" / f"recorded-ozgenel-test-{STAMP}.json"
out = infer.main(["--seg", str(SEG), "--cls", str(CLS), "--images", "data/raw/ozgenel_seg/extracted",
                  "--list", "data/splits/ozgenel_seg/test.txt", "--device", "auto", "--out", str(test_cache)])
print(out["provenance"]); print(out["summary"])

In [ ]:
ex_cache = REPO / "results" / f"recorded-examples-{STAMP}.json"
out = infer.main(["--seg", str(SEG), "--cls", str(CLS), "--images", "examples", "--device", "auto",
                  "--out", str(ex_cache)])
print(out["summary"])